# L07 Assignment — Two PyTorch Exercises

Apply the L07 toolkit (PyTorch `Dataset` + `DataLoader` + `nn.Module` + training loop) to two new domains.

1. **Part A — Loan default prediction.** Binary classification. Predict whether a loan will default.
2. **Part B — California housing price.** Regression. Predict the median house value.

**Time budget:** ~75 minutes.

---

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)
print("✅ Setup complete.")

✅ Setup complete.


---

## 📚 Choose your track

This assignment has **two tracks**. Pick **one** based on your background — you don't need to do both.

| Track | Who it's for | What you'll do |
|---|---|---|
| **🟢 Foundational Track** | Learners new to ML / programming | Part A — loan-default binary classification with PyTorch |
| **🔵 Advanced Track** | Learners with prior ML background | Part B — California housing regression (adapt the loop) |

If you're unsure, start with the **Foundational Track**. If it feels easy, skip ahead to the **Advanced Track** — both tracks cover the same lesson outcomes; only the scaffolding differs.

---


---

# 🟢 Foundational Track

> *No prior ML background needed. The cells below are scaffolded — read the worked example, then fill in the blanks. Hints are included.*

---


# Part A — Loan Default Prediction

**Why this problem matters:** every loan a bank approves that later defaults is money lost; every good customer it rejects is revenue missed. Predicting default risk from application data is one of the oldest, highest-stakes prediction problems in business — and it's a binary yes/no question, exactly like Sarah's checkout-completion task.

We'll use a synthetic loan dataset (it mirrors the structure of real consumer-credit data, with no privacy issues).

In [2]:
# Build a fake-but-realistic loan book: each row = one loan, each column = one applicant fact.
# You don't need to follow the maths here — just run it and look at the table it produces.
def generate_loan_data(n=10000, seed=2026):
    rng = np.random.default_rng(seed)
    income = np.round(rng.lognormal(mean=10.7, sigma=0.5, size=n).clip(15000, 250000), 0)
    loan_amount = np.round(rng.lognormal(mean=10.2, sigma=0.6, size=n).clip(1000, 100000), 0)
    age = rng.integers(20, 70, size=n)
    credit_score = rng.normal(680, 90, size=n).clip(300, 850).round()
    debt_to_income = (loan_amount / income).clip(0, 5).round(2)
    employment_years = rng.exponential(scale=6, size=n).clip(0, 40).round(1)
    has_mortgage = rng.binomial(1, 0.55, size=n)
    n_credit_lines = rng.integers(1, 15, size=n)
    home_owner = rng.binomial(1, 0.62, size=n)

    # Default risk — deliberately non-linear, with interactions (e.g. high debt AND low
    # credit score together are worse than either alone) so an MLP has something to find.
    logit = (-2.5
             - 0.008 * (credit_score - 680)
             + 1.5 * debt_to_income
             + 0.05 * (40 - employment_years)
             - 0.5 * home_owner
             + 0.5 * (debt_to_income > 0.4).astype(float) * (credit_score < 650).astype(float)
             + rng.normal(0, 0.4, size=n))
    prob = 1 / (1 + np.exp(-logit))
    default = (rng.random(n) < prob).astype(int)

    return pd.DataFrame({
        "loan_id": np.arange(100000, 100000 + n),
        "income": income, "loan_amount": loan_amount, "age": age,
        "credit_score": credit_score, "debt_to_income": debt_to_income,
        "employment_years": employment_years, "has_mortgage": has_mortgage,
        "n_credit_lines": n_credit_lines, "home_owner": home_owner,
        "default": default,
    })

loans = generate_loan_data()
print(f"Loan dataset: {len(loans):,} loans · default rate: {loans['default'].mean():.1%}")
loans.head()

Loan dataset: 10,000 loans · default rate: 53.5%


,loan_id,income,loan_amount,age,credit_score,debt_to_income,employment_years,has_mortgage,n_credit_lines,home_owner,default
0,100000,29835.0,24752.0,66,525.0,0.83,6.6,1,12,0,1
1,100001,50025.0,10776.0,53,850.0,0.22,36.1,0,4,1,0
2,100002,17186.0,10391.0,59,747.0,0.60,10.9,0,1,0,0
3,100003,89133.0,25987.0,30,636.0,0.29,0.1,1,10,1,1
4,100004,61032.0,59577.0,36,708.0,0.98,12.1,0,14,1,0


## Exercise A1 — Prepare the data

**Tasks:**
1. Define X (drop loan_id and default) and y (default).
2. 80/20 train/test split (stratified).
3. Standardise X with StandardScaler.
4. Convert all to PyTorch tensors.

<details>
<summary>💡 Hint 1 — where you've done this before</summary>

This is exactly Step 1 of `04_pytorch_training_loop.ipynb` (cell "Prepare data + tensors"), with `loans`/`default` instead of sessions/`completed`. Open it side by side and adapt.
</details>

<details>
<summary>💡 Hint 2 — the skeleton</summary>

```python
y = loans["default"].values
X = loans.drop(columns=["loan_id", "default"]).values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)   # fit on train ONLY
X_te_s = sc.transform(X_te)      # apply the same scaling to test
# then torch.tensor(..., dtype=torch.float32) for each of the four arrays
```
</details>

*Your code:*

In [3]:
# Exercise A1
# (your code here)


## Exercise A2 — Build the PyTorch model

**Task:** define a `LoanMLP` class with TWO hidden layers (64 → 32 neurons), ReLU activations, single logit output.

<details>
<summary>💡 Hint — the pattern to copy</summary>

Copy the `MLP` class from `04_pytorch_training_loop.ipynb` Step 3 and change the layer sizes: `nn.Linear(n_features, 64) → ReLU → nn.Linear(64, 32) → ReLU → nn.Linear(32, 1)`. No sigmoid at the end — `BCEWithLogitsLoss` expects the raw logit.
</details>

*Your code:*

In [4]:
# Exercise A2
# (your code here)


## Exercise A3 — Train with the standard loop

**Tasks:**
1. Wrap train data in `TensorDataset` + `DataLoader(batch_size=64, shuffle=True)`.
2. Use `BCEWithLogitsLoss` + `Adam(lr=1e-3)`.
3. Train 30 epochs (or use early stopping with patience=5).
4. Report test AUC.

<details>
<summary>💡 Hint 1 — the five lines, always the same</summary>

Inside the batch loop it is always: `zero_grad → forward → loss → backward → step`. Wrap that in `for epoch in range(30):` and `for X_b, y_b in loader:`. Remember `.squeeze(-1)` on the model output so its shape matches `y_b`.
</details>

<details>
<summary>💡 Hint 2 — scoring at the end</summary>

```python
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_te_t).squeeze(-1)).numpy()  # logits → probabilities
print(roc_auc_score(y_te, probs))
```
Early stopping is optional here — 30 fixed epochs is fine. If you want it, reuse `train_pytorch()` from notebook 04.
</details>

*Your code:*

In [5]:
# Exercise A3
# (your code here)


---

# 🔵 Advanced Track

> *For learners with prior ML background. Minimal scaffolding — you decide the approach. You're welcome to peek at the Foundational Track above for reference.*

---


# Part B — California Housing Price (Regression)

**Why this exercise:** the claim all week has been that the five-line training loop is universal. Here's the test — predicting a *number* (house price) instead of a yes/no. If you can make the two small swaps below yourself, you've truly learned the pattern, not memorised one script.

Switch from binary classification to regression. The PyTorch loop is nearly identical — only differences: `MSELoss` instead of `BCEWithLogitsLoss`, and the model outputs a real number instead of a logit.

In [6]:
# Generate a synthetic California-style housing dataset — each row is one neighbourhood;
# the target `median_value_100k` is the median house value in units of $100k.
def generate_housing_data(n=8000, seed=2027):
    rng = np.random.default_rng(seed)
    median_income       = rng.lognormal(mean=1.5, sigma=0.4, size=n).clip(0.5, 15).round(2)
    house_age           = rng.integers(1, 60, size=n)
    avg_rooms           = rng.normal(5.5, 1.5, size=n).clip(2, 12).round(1)
    avg_bedrooms        = (avg_rooms * rng.uniform(0.15, 0.30, size=n)).round(1)
    population          = rng.lognormal(mean=7.0, sigma=0.6, size=n).clip(100, 30000).round()
    households          = (population * rng.uniform(0.3, 0.5, size=n)).round()
    distance_to_coast_km = rng.exponential(scale=80, size=n).clip(1, 500).round(0)

    # Median house value (in $100k)
    value = (
        1.0 + 0.45 * median_income                       # income strong driver
        + 0.02 * avg_rooms
        - 0.01 * house_age                                # newer slightly less expensive (paradox of CA data!)
        + 0.005 * (median_income * avg_rooms)             # interaction
        - 0.002 * distance_to_coast_km                    # coastal premium
        + rng.normal(0, 0.4, size=n)
    ).clip(0.3, 10).round(2)

    return pd.DataFrame({
        "house_id": np.arange(200000, 200000 + n),
        "median_income": median_income, "house_age": house_age,
        "avg_rooms": avg_rooms, "avg_bedrooms": avg_bedrooms,
        "population": population, "households": households,
        "distance_to_coast_km": distance_to_coast_km,
        "median_value_100k": value,
    })

housing = generate_housing_data()
# Values are stored in units of $100k, so multiply by 100 to print in $k.
print(f"Housing dataset: {len(housing):,} rows · median value range: ${housing['median_value_100k'].min()*100:.0f}k - ${housing['median_value_100k'].max()*100:.0f}k")
housing.head()

Housing dataset: 8,000 rows · median value range: $0.300k - $8.2100k


,house_id,median_income,house_age,avg_rooms,avg_bedrooms,population,households,distance_to_coast_km,median_value_100k
0,200000,4.68,6,7.0,1.6,902.0,390.0,30.0,3.36
1,200001,4.33,2,5.9,1.7,3095.0,1517.0,42.0,2.88
2,200002,3.25,57,2.0,0.4,483.0,160.0,43.0,1.77
3,200003,1.89,3,5.3,1.5,1869.0,569.0,10.0,2.42
4,200004,7.28,35,4.4,1.1,992.0,436.0,234.0,3.56


## Exercise B1 — Adapt the training loop for regression

**Tasks:**
1. Define X (drop house_id and median_value_100k) and y (median_value_100k).
2. Train/test split + scale X.
3. Build a regression MLP — same `nn.Sequential` pattern as classification, but output ONE neuron with NO sigmoid (raw real-valued output).
4. Loss function: `nn.MSELoss()` (mean squared error).
5. Train with Adam + batches.
6. Report **test MAE** in the original units ($100k).

<details>
<summary>💡 Hint — the only two things that change from Part A</summary>

1. **Loss:** `criterion = nn.MSELoss()` — average squared gap between predicted and true price.
2. **No sigmoid anywhere:** the output neuron IS the predicted price (a raw number).

Everything else — split, scale, tensors, `DataLoader`, `Adam`, the five-line loop — is identical to A1–A3. One extra trick worth doing: standardise **y** too (`(y - mean) / std`) so the loss starts at a sane scale, then un-scale predictions before computing MAE.
</details>

*Your code:*

In [7]:
# Exercise B1
# (your code here)


*Your interpretation: what test MAE would you consider 'good' here? (Hint: target std)*

> (your answer here)

## ✅ Submission checklist

- [ ] Exercise A1: data prepared as PyTorch tensors
- [ ] Exercise A2: LoanMLP class with two hidden layers
- [ ] Exercise A3: trained loop + test AUC reported
- [ ] Exercise B1: regression MLP + test MAE reported + interpretation

---

# 📚 Sample solutions

## Sample — Exercises A1 + A2 + A3

In [8]:
# === Sample Part A ===

# --- A1: prepare the data ---

y = loans["default"].values                          # label: did the loan default? (1/0)
X = loans.drop(columns=["loan_id", "default"]).values   # features: drop id + the answer

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)
sc = StandardScaler()                                 # rescale to mean 0 / std 1 (nets train better)
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)   # fit on train only — no test peeking

# Convert NumPy arrays to PyTorch tensors (the format the model expects).
X_tr_t = torch.tensor(X_tr_s, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
X_te_t = torch.tensor(X_te_s, dtype=torch.float32)
y_te_t = torch.tensor(y_te, dtype=torch.float32)

# --- A2: the model — 9 inputs → 64 → 32 → 1 raw score (logit) ---
class LoanMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x): return self.layers(x)

# --- A3: train with the standard loop ---
torch.manual_seed(42)                                 # reproducible starting weights
model_loan = LoanMLP(X_tr_t.shape[1])
loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)   # shuffled 64-row batches
crit = nn.BCEWithLogitsLoss()                         # loss for binary yes/no from raw logits
opt = optim.Adam(model_loan.parameters(), lr=1e-3)    # the weight-tuner; 1e-3 is the safe default

for epoch in range(30):                               # 30 full passes over the training data
    model_loan.train()                                # training mode
    for X_b, y_b in loader:                           # one mini-batch at a time
        opt.zero_grad()                               # 1. clear old gradients
        l = crit(model_loan(X_b).squeeze(-1), y_b)    # 2-3. forward pass + how wrong were we?
        l.backward(); opt.step()                      # 4-5. compute gradients, step downhill

model_loan.eval()                                     # evaluation mode
with torch.no_grad():                                 # just predicting — no gradient tracking
    probs = torch.sigmoid(model_loan(X_te_t).squeeze(-1)).numpy()   # logits → 0-1 probabilities
auc = roc_auc_score(y_te, probs)
print(f"Loan-default MLP test AUC: {auc:.3f}")

Loan-default MLP test AUC: 0.771


## Sample — Exercise B1 (regression)

In [9]:
# === Sample Part B ===

# --- Prepare data (same recipe as Part A, but the target is a NUMBER, not 0/1) ---

y_h = housing["median_value_100k"].values
X_h = housing.drop(columns=["house_id", "median_value_100k"]).values

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_h, y_h, test_size=0.20, random_state=42)
sc_h = StandardScaler()
X_tr_hs = sc_h.fit_transform(X_tr_h); X_te_hs = sc_h.transform(X_te_h)

# Also scale the target for stable training (we'll unscale predictions)
y_mean, y_std = y_tr_h.mean(), y_tr_h.std()

X_tr_ht = torch.tensor(X_tr_hs, dtype=torch.float32)
y_tr_ht = torch.tensor((y_tr_h - y_mean) / y_std, dtype=torch.float32)
X_te_ht = torch.tensor(X_te_hs, dtype=torch.float32)
y_te_ht = torch.tensor((y_te_h - y_mean) / y_std, dtype=torch.float32)

# --- Model: identical shape to LoanMLP — the 1 output neuron is now a price, not a logit ---
class HousingMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x): return self.layers(x)

torch.manual_seed(42)
model_h = HousingMLP(X_tr_ht.shape[1])
loader_h = DataLoader(TensorDataset(X_tr_ht, y_tr_ht), batch_size=64, shuffle=True)
crit_h = nn.MSELoss()                    # regression loss: average squared prediction error
opt_h = optim.Adam(model_h.parameters(), lr=1e-3)

# The SAME five-line loop as every other model this week — only the loss changed.
for epoch in range(40):
    model_h.train()
    for X_b, y_b in loader_h:
        opt_h.zero_grad()                             # clear old gradients
        l = crit_h(model_h(X_b).squeeze(-1), y_b)     # forward + measure squared error
        l.backward(); opt_h.step()                    # gradients, then one downhill step

model_h.eval()
with torch.no_grad():
    preds_scaled = model_h(X_te_ht).squeeze(-1).numpy()
preds = preds_scaled * y_std + y_mean   # unscale back to $100k units
mae = mean_absolute_error(y_te_h, preds)
print(f"Housing MLP test MAE: ${mae * 100:.1f}k")
print(f"Target std: ${y_h.std() * 100:.1f}k → so MAE / std = {mae/y_h.std():.2f}")

Housing MLP test MAE: $31.9k
Target std: $107.4k → so MAE / std = 0.30


**Interpretation:** the sample run lands around test MAE ≈ $32k against a target std of ~$107k — an MAE/std ratio of ~0.30. As a rough rule of thumb, MAE / target-std under 0.7 means the model is "useful" (clearly better than always predicting the average); the lower the ratio, the better.

## What's next

You've now used PyTorch on:
- Binary classification (L07 in-class — session completion)
- Loan default classification (this assignment, Part A)
- Housing price regression (this assignment, Part B)

The training loop is THE SAME for all three. The differences:
- Loss function (`BCEWithLogitsLoss` for binary, `CrossEntropyLoss` for multiclass, `MSELoss` for regression)
- Output layer size (1 for binary/regression, K for K-class classification)

**Next session → L08 (Computer Vision).** Marcus's question — *"automatically tag product PHOTOS as dress, jeans, jacket?"* — moves us from tabular data to IMAGES. Same PyTorch training loop, new architecture (CNN).